In [1]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path


In [2]:
BASE_DIR = Path.cwd()
NETWORK_DIR = BASE_DIR / "cytoscape_networks"
GROWTH_PLOT_DIR = BASE_DIR / "paper_figs"
PLOT_FIG_TYPE = "png"  # can be "png" or "svg"
FIG_WIDTH_IN = 7.5  # full page width

figures = {
    "main_ah_bi"                : {
        "cyto_network": NETWORK_DIR / "ahallii_binfantis_Exchanges.png",
        "growth_plot": GROWTH_PLOT_DIR / "2_ahallii_binfantis.png",
        "outfile": BASE_DIR / "paper_figs" / f"AH_BI_main_figure.{PLOT_FIG_TYPE}"
    },
}


def crop_whitespace(img: np.ndarray, pad: int = 10, thresh: float = 0.98) -> np.ndarray:
    """Crop near-white or transparent borders from an RGB(A) image array."""
    rgb = img[..., :3]
    content = rgb.min(axis=2) < thresh
    if img.shape[2] == 4:
        content &= img[..., 3] > 0.02
    rows = np.where(content.any(axis=1))[0]
    cols = np.where(content.any(axis=0))[0]
    if rows.size == 0:
        return img
    r0, r1 = max(rows[0] - pad, 0), min(rows[-1] + pad + 1, img.shape[0])
    c0, c1 = max(cols[0] - pad, 0), min(cols[-1] + pad + 1, img.shape[1])
    return img[r0:r1, c0:c1]


for figure in figures.keys():
    outfile = figures[figure]["outfile"]

    network = crop_whitespace(plt.imread(figures[figure]["cyto_network"]))
    growth = crop_whitespace(plt.imread(figures[figure]["growth_plot"]))

    network_h, network_w = network.shape[:2]
    growth_h, growth_w = growth.shape[:2]

    # width ratios proportional to each image's aspect ratio -> equal panel heights
    network_ratio = network_w / network_h
    growth_ratio = growth_w / growth_h
    fig_height = FIG_WIDTH_IN / (network_ratio + growth_ratio)

    fig, axs = plt.subplots(
        nrows=1,
        ncols=2,
        figsize=(FIG_WIDTH_IN, fig_height),
        gridspec_kw={"width_ratios": [network_ratio, growth_ratio], "wspace": 0.02},
        squeeze=False,
    )
    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)

    for ax, img, letter in zip(axs[0], (network, growth), ("A", "B")):
        ax.imshow(img, interpolation="lanczos")
        ax.axis("off")
        ax.text(0.0, 1.0, letter, transform=ax.transAxes,
                fontsize=12, fontweight="bold", va="top", ha="left")

    plt.savefig(outfile, bbox_inches="tight", dpi=600)
    plt.close(fig)
